<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session07-discussion-4.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 7: in-class discussion problem for block 3

This is the block 3 question of the lecture quiz. First guess the answer in the quiz, on your own.
Then fix and run the code below, discuss the outcome with your neighbour, and answer the quiz again.

## Picture 1 or picture 2?

Today's two pictures: **picture 1** is one weighted sum of the inputs, perhaps followed by an
activation at the output; **picture 2** has a hidden layer with a nonlinear activation between two
weighted sums. Which of these methods are picture 2: PSSM, perceptron, logistic regression, GOR, MLP,
PHD?

**A test you can run.** A transcription factor binds a six-base site only when the second base pairs
with the first (A with T, G with C), as in Bulyk et al. (2002): the positions are not independent. Each
base on its own says nothing. The cell below makes 2000 such sites, half of them bound, and trains
four models on the one-hot encoded sites.

**Fix and run.** Run it once as it is: which models learn the rule? The line marked `FIX` gives the
MLP no nonlinear activation in its hidden layer. Change `"identity"` to `"relu"` and run it again.

In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

rng = np.random.default_rng(0)
pair = {"A": "T", "T": "A", "G": "C", "C": "G"}

def site(bound):
    s = list(rng.choice(list("ACGT"), 6))
    others = [b for b in "ACGT" if b != pair[s[0]]]
    s[1] = pair[s[0]] if bound else rng.choice(others)
    return "".join(s)

labels = rng.integers(0, 2, 2000)
seqs = [site(b) for b in labels]
X = np.array([[c == b for c in s for b in "ACGT"] for s in seqs], dtype=float)   # one-hot, 6 × 4
X_tr, X_te, y_tr, y_te = train_test_split(X, labels, test_size=0.25, random_state=0)
print("first four sites:", seqs[:4], "bound:", labels[:4])

# A PSSM: one log-odds score per position and base, summed over the site
pssm = np.log((X_tr[y_tr == 1].sum(0) + 1) / (X_tr[y_tr == 0].sum(0) + 1))
print(f"{'PSSM':22s} test accuracy {np.mean((X_te @ pssm > 0) == y_te):.2f}")

models = {
    "perceptron": Perceptron(),
    "logistic regression": LogisticRegression(max_iter=1000),
    "MLP, 16 hidden units": MLPClassifier(hidden_layer_sizes=(16,), activation="identity",   # FIX
                                          max_iter=3000, random_state=0),
}
for name, model in models.items():
    model.fit(X_tr, y_tr)
    print(f"{name:22s} test accuracy {model.score(X_te, y_te):.2f}")

first four sites: ['GCGTTC', 'CGGGTT', 'GCGCCA', 'CAGACA'] bound: [1 1 1 0]
PSSM                   test accuracy 0.52
perceptron             test accuracy 0.55
logistic regression    test accuracy 0.52
MLP, 16 hidden units   test accuracy 0.52


**If you finish early.**

1. GOR scores a residue by adding one log-odds term per neighbouring residue. Which line in the
   cell does the same kind of thing?
2. PHD feeds a window of 13 residues into a hidden layer with a sigmoid. Would it learn this rule?

**Discussion point.** As it was, all four models scored about 0.5: no better than guessing.
With `activation="relu"`, the MLP scores 1.00, and the others stay at 0.5.

The PSSM, the perceptron and logistic regression are all picture 1: one weighted sum of the inputs,
one score per position and base. An activation at the output (the perceptron's step, logistic
regression's sigmoid) only turns that sum into a decision; it does not add a hidden layer. Picture 1
can only add up evidence from each base on its own, and here each base on its own carries none. GOR
is picture 1 too: a sum of log-odds terms, like the PSSM.

The MLP without an activation was still picture 1: its two layers multiply out to one weighted sum
(block 2). With ReLU between them it becomes picture 2, and it can learn that two positions must
match. PHD is picture 2 as well: a window, a hidden layer with a sigmoid, and the outputs. So the
quiz answer is MLP and PHD.